In [1]:
from openff.toolkit import Molecule, Topology, ForceField

In [2]:
# load most recent SMIRNOFF small molecule force field (has TIP3P water model)
sage = ForceField("openff-2.2.1.offxml")

/Users/mattthompson/.local/share/mamba/envs/openff-toolkit-test-rdkit/lib/python3.12/site-packages/openff/amber_ff_ports/amber_ff_ports.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [3]:
# load each molecule of the dimer (`Molecule`s), condense into single representation (a `Topology`)
topology = Topology.from_molecules(
    [
        Molecule.from_file("bromobenzene.sdf", file_format="sdf"),
        Molecule.from_file("water.sdf", file_format="sdf"),
    ]
)

In [4]:
# apply the force field, get an `Interchange` out
interchange = sage.create_interchange(topology)

In [5]:
# visualize the interchange - can also do topology.visuzlize() which uses similar code
interchange.visualize()

NGLWidget()

In [6]:
# save out the original coordinates into a variable
initial_coordinates = interchange.positions
initial_coordinates

Magnitude,[[0.0 0.0 0.181439995765686] [0.0 0.0 -0.010329999774694442] [0.0 0.12180999517440795 -0.0786899983882904] [0.0 -0.12180999517440795 -0.0786899983882904] [0.0 0.12092000246047972 -0.2185199975967407] [0.0 -0.12092000246047972 -0.2185199975967407] [0.0 0.0 -0.2886699914932251] [0.0 0.21584000587463378 -0.023440000414848325] [0.0 -0.21584000587463378 -0.023440000414848325] [0.0 0.21584999561309812 -0.2725800037384033] [0.0 -0.21584999561309812 -0.2725800037384033] [0.0 0.0 -0.39786999225616454] [0.0 0.16000000238418577 0.45857000350952143] [-0.03318000137805938 0.11441999673843382 0.37962999343872067] [0.007129999995231628 0.2515599966049194 0.4283299922943115]]
Units,nanometer


In [7]:
# minimize using Sage; this method wraps OpenMM and updates the positions attribute
# with the new coordinates - see docs for more
# https://docs.openforcefield.org/projects/interchange/en/stable/_autosummary/openff.interchange.Interchange.html#openff.interchange.Interchange.minimize
interchange.minimize()

In [8]:
# save out the "optimized" coordinates
optimized_coordinates = interchange.positions
optimized_coordinates

Magnitude,[[0.013977278023958206 0.020546400919556618 0.17707791924476624] [0.0066436599008738995 0.010277151130139828 -0.01354711502790451] [-0.0013505511451512575 0.12829561531543732 -0.09036780893802643] [0.009189199656248093 -0.11539804190397263 -0.07750047743320465] [-0.006836854387074709 0.12071499973535538 -0.23124197125434875] [0.003705885959789157 -0.12318289279937744 -0.21836787462234497] [-0.004312833305448294 -0.005070360377430916 -0.29524269700050354] [-0.0032878206111490726 0.22481565177440643 -0.0406382791697979] [0.015353808179497719 -0.20591293275356293 -0.017818402498960495] [-0.012969424948096275 0.21164962649345398 -0.29028618335723877] [0.005691182799637318 -0.22001148760318756 -0.2674923837184906] [-0.008525696583092213 -0.010986410081386566 -0.40359488129615784] [-0.0244810339063406 0.1840248852968216 0.4771173596382141] [-0.08190149068832397 0.1375618577003479 0.4162371754646301] [0.06384653598070145 0.1675494909286499 0.444115549325943]]
Units,nanometer


In [12]:
# visualize again - looks like the water has rotated somewhat significantly
interchange.visualize()

NGLWidget()

In [10]:
# can do whatever math on these coordinates, they're just Nx3 arrays at each state
import numpy

print(f"Coordinate differences = {numpy.round(optimized_coordinates - initial_coordinates, 3)}")

Coordinate differences = [[0.014 0.021 -0.004] [0.007 0.01 -0.003] [-0.001 0.006 -0.012] [0.009 0.006 0.001] [-0.007 -0.0 -0.013] [0.004 -0.002 0.0] [-0.004 -0.005 -0.007] [-0.003 0.009 -0.017] [0.015 0.01 0.006] [-0.013 -0.004 -0.018] [0.006 -0.004 0.005] [-0.009 -0.011 -0.006] [-0.024 0.024 0.019] [-0.049 0.023 0.037] [0.057 -0.084 0.016]] nanometer


In [11]:
# for example, which atom coordinates (rows) moved by more than 0.3 Angstroms? (only the water atoms, listed last)
(numpy.sum((optimized_coordinates - initial_coordinates) ** 2, axis=1) ** 0.5).m_as("angstrom") > 0.3

array([False, False, False, False, False, False, False, False, False,
       False, False, False,  True,  True,  True])